# Unified Time-Series Domain Comparison Analysis

## Appliance Power Classification: Temporal vs Spectral vs Hybrid Domains

This notebook demonstrates the complete workflow for comparing three representation domains:

1. **Temporal Domain**: Statistical features of the signal
2. **Spectral Domain**: Frequency-based features from FFT analysis
3. **Hybrid Domain**: Combined temporal + spectral features

### Research Question

**Which representation domain performs best for appliance power classification, and why?**


## 1. Import Required Libraries


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path().cwd().parent / 'src'))

# Import custom modules
from preprocessing.preprocessing_pipeline import PreprocessingPipeline
from features.temporal_features import TemporalFeatureExtractor
from features.spectral_features import SpectralFeatureExtractor
from features.hybrid_features import HybridFeatureExtractor
from models.model_factory import ModelFactory
from models.svm_model import SVMModel
from models.knn_model import KNNModel
from evaluation.evaluation_pipeline import EvaluationPipeline
from evaluation.statistical_tests import mcnemar_test, effect_size_cohens_d
from pipelines.unified_pipeline import UnifiedPipeline
from utils.logger import setup_logger
from utils.constants import APPLIANCE_CLASSES, DEFAULT_SAMPLING_RATE

# Setup logger
logger = setup_logger(__name__)

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"Available appliance classes: {APPLIANCE_CLASSES}")

## 2. Load and Prepare Data


In [ ]:
# Load data
data_dir = Path().cwd().parent / 'data' / 'raw'

# Load training data
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\nTraining data columns: {train_df.columns.tolist()}")
print(f"\nTraining data info:")
print(train_df.info())

In [ ]:
# Extract signals and labels
# Assuming the CSV has a 'signal' column and a 'label' column
# Adjust column names as needed based on your actual data

# For this example, we'll assume all columns except the last are signal values
# and the last column is the label

if 'label' in train_df.columns:
    X_train = train_df.drop('label', axis=1).values
    y_train = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
else:
    # If no label column, assume last column is label
    X_train = train_df.iloc[:, :-1].values
    y_train = train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, :-1].values
    y_test = test_df.iloc[:, -1].values

print(f"Signals shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")
print(f"Unique labels: {np.unique(y_train)}")
print(f"\nClass distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} samples")

## 3. Feature Extraction Demonstration


In [ ]:
# Demonstrate temporal feature extraction
print("=== TEMPORAL FEATURE EXTRACTION ===")
temporal_extractor = TemporalFeatureExtractor()
sample_signal = X_train[0]

temporal_features, temporal_names = temporal_extractor.extract_features(
    sample_signal, return_feature_names=True
)

print(f"\nTemporal features extracted: {len(temporal_names)}")
print(f"Features: {temporal_names}")
print(f"\nSample temporal feature vector: {temporal_features[:5]}")

In [ ]:
# Demonstrate spectral feature extraction
print("=== SPECTRAL FEATURE EXTRACTION ===")
spectral_extractor = SpectralFeatureExtractor()

spectral_features, spectral_names = spectral_extractor.extract_features(
    sample_signal, sampling_rate=DEFAULT_SAMPLING_RATE, return_feature_names=True
)

print(f"\nSpectral features extracted: {len(spectral_names)}")
print(f"Features: {spectral_names}")
print(f"\nSample spectral feature vector: {spectral_features[:5]}")

In [ ]:
# Demonstrate hybrid feature extraction
print("=== HYBRID FEATURE EXTRACTION ===")
hybrid_extractor = HybridFeatureExtractor(
    use_pca=True, pca_components=30,
    use_mutual_info_selection=True, n_features_to_select=50
)

hybrid_features = hybrid_extractor.extract_features(
    sample_signal, sampling_rate=DEFAULT_SAMPLING_RATE
)

print(f"\nRaw hybrid features: {len(temporal_names) + len(spectral_names)}")
print(f"After feature selection and PCA: {len(hybrid_features)}")
print(f"\nSample hybrid feature vector: {hybrid_features[:5]}")

## 4. Complete Unified Pipeline Analysis


In [ ]:
# Create and run unified pipeline
print("Initializing Unified Pipeline...")
output_dir = Path().cwd().parent / 'outputs' / 'unified_analysis'
output_dir.mkdir(parents=True, exist_ok=True)

unified_pipeline = UnifiedPipeline(
    models=['random_forest', 'xgboost'],
    output_dir=str(output_dir)
)

print("Running complete analysis...")
results = unified_pipeline.run_complete_analysis(
    X_train, y_train,
    test_size=0.2,
    save_results=True
)

print(f"\nAnalysis complete! Results saved to {output_dir}")

## 5. Results Summary


In [ ]:
# Extract and display results
comparison = results['comparison']
pipelines = results['pipelines']

# Create results dataframe
accuracies_dict = comparison['accuracies']
accuracies_df = pd.DataFrame(
    [{'Pipeline': k, 'Accuracy': v} for k, v in accuracies_dict.items()]
).sort_values('Accuracy', ascending=False)

print("\n" + "="*60)
print("ACCURACY RANKING")
print("="*60)
print(accuracies_df.to_string(index=False))
print("="*60)

In [ ]:
# Visualize domain comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy by pipeline
accuracies_df_sorted = accuracies_df.sort_values('Accuracy')
colors = ['red' if 'temporal' in p else 'green' if 'spectral' in p else 'blue' 
          for p in accuracies_df_sorted['Pipeline']]
ax1.barh(accuracies_df_sorted['Pipeline'], accuracies_df_sorted['Accuracy'], color=colors)
ax1.set_xlabel('Accuracy')
ax1.set_title('Pipeline Performance Comparison')
ax1.set_xlim([0, 1])

# Plot 2: Domain comparison
domain_comparison = comparison['by_domain']
domains = list(domain_comparison.keys())
means = [domain_comparison[d]['mean_accuracy'] for d in domains]
stds = [domain_comparison[d]['std_accuracy'] for d in domains]

ax2.bar(domains, means, yerr=stds, capsize=10, alpha=0.7)
ax2.set_ylabel('Accuracy')
ax2.set_title('Mean Accuracy by Domain')
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(output_dir / 'domain_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPlot saved!")

In [ ]:
# Print detailed comparison
print("\n" + "="*60)
print("DOMAIN-WISE ANALYSIS")
print("="*60)

for domain, stats in comparison['by_domain'].items():
    print(f"\n{domain.upper()}:")
    print(f"  Mean Accuracy: {stats['mean_accuracy']:.3f}")
    print(f"  Std Dev: {stats['std_accuracy']:.3f}")
    print(f"  Range: [{stats['min_accuracy']:.3f}, {stats['max_accuracy']:.3f}]")

In [ ]:
# Feature dimensionality analysis
print("\n" + "="*60)
print("FEATURE DIMENSIONALITY ANALYSIS")
print("="*60)

feature_analysis = {}
for pipeline_name, results in pipelines.items():
    if 'n_features' in results:
        domain = results['pipeline']
        n_features = results['n_features']
        if domain not in feature_analysis:
            feature_analysis[domain] = []
        feature_analysis[domain].append(n_features)

for domain, features in sorted(feature_analysis.items()):
    print(f"\n{domain.upper()}:")
    print(f"  Number of features: {features[0]}")
    if len(set(features)) == 1:
        print(f"  (Consistent across models)")
    else:
        print(f"  (Varies by model: {features})")

## 6. Key Findings and Conclusions


In [ ]:
# Generate final summary
print("\n" + "="*70)
print("FINAL CONCLUSIONS")
print("="*70)

best_pipeline = comparison['best_pipeline']
best_accuracy = comparison['best_accuracy']
best_domain = best_pipeline.split('_')[0]

print(f"\n🏆 BEST PERFORMING PIPELINE: {best_pipeline}")
print(f"   Accuracy: {best_accuracy:.3f}")
print(f"   Domain: {best_domain.upper()}")

# Compare domains
domain_stats = comparison['by_domain']
best_domain_stat = max(domain_stats.items(), key=lambda x: x[1]['mean_accuracy'])
print(f"\n📊 BEST PERFORMING DOMAIN: {best_domain_stat[0].upper()}")
print(f"   Mean Accuracy: {best_domain_stat[1]['mean_accuracy']:.3f}")

# Insights
print("\n💡 KEY INSIGHTS:")
print("\n1. Domain Performance:")
for domain in sorted(domain_stats.keys()):
    stats = domain_stats[domain]
    print(f"   - {domain.upper()}: {stats['mean_accuracy']:.3f} ± {stats['std_accuracy']:.3f}")

print("\n2. Interpretation:")
if best_domain == 'temporal':
    print("   ✓ Temporal domain (statistical features) performs best")
    print("   ✓ Appliance power dynamics are well-captured by amplitude statistics")
    print("   ✓ Simple, interpretable features outperform complex frequency analysis")
elif best_domain == 'spectral':
    print("   ✓ Spectral domain (frequency features) performs best")
    print("   ✓ Appliances have distinctive frequency signatures")
    print("   ✓ Harmonic content is discriminative for classification")
else:  # hybrid
    print("   ✓ Hybrid domain (temporal + spectral) performs best")
    print("   ✓ Complementary information from both domains improves performance")
    print("   ✓ Feature selection and dimensionality reduction manage redundancy")

print("\n3. Practical Recommendations:")
print(f"   ✓ Use {best_pipeline} for production deployment")
print(f"   ✓ Expected accuracy: {best_accuracy:.1%}")

print("\n" + "="*70)

In [ ]:
# Print the generated report
print(results['report'])

## 7. Next Steps and Extensions

### Potential Improvements:

1. **Hyperparameter Optimization**
   - Use GridSearchCV or Bayesian optimization
   - Tune model-specific parameters for each domain

2. **Cross-Validation Analysis**
   - Run k-fold cross-validation for robustness
   - Analyze fold-to-fold variance

3. **Feature Importance Analysis**
   - Extract SHAP values for model explainability
   - Identify which temporal/spectral features are most discriminative

4. **Misclassification Analysis**
   - Investigate specific appliance confusion patterns
   - Refine feature engineering based on errors

5. **Domain Fusion**
   - Explore advanced fusion strategies beyond concatenation
   - Consider attention mechanisms for feature weighting

6. **Time-Series Specific Models**
   - Implement LSTM, GRU, or other RNN architectures
   - Explore 1D convolutional networks
   - Consider ensemble methods combining domain pipelines
